In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.\
            builder.\
                appName('Sort Union Aggregation').\
                    getOrCreate()

In [0]:
emp_schema = "id INT, name STRING, department STRING, salary DOUBLE, hire_date STRING, country STRING, age INT, gender STRING"

emp_data = [
    (1, "John Smith", "Engineering", 85000.0, "2021-03-15", "USA", 30, "Male"),
    (2, "Alice Johnson", "Engineering", 92000.0, "2020-07-01", "USA", 35, "Female"),
    (3, "Bob Williams", "Sales", 65000.0, "2022-01-20", "UK", 28, None),
    (4, "Carol Brown", "Sales", 68000.0, "2019-11-05", "UK", 40, "Female"),
    (5, "David Jones", "HR", 55000.0, "2021-09-10", "Canada", 32, "Male"),
    (6, "Eva Garcia", "HR", 58000.0, "2023-02-28", "Canada", 26, "Female"),
    (7, "Frank Miller", "Engineering", 78000.0, "2018-05-22", "Germany", 45, "Male"),
    (8, "Grace Davis", "Marketing", 72000.0, "2022-08-14", "Germany", 29, "Female"),
    (9, "Henry Wilson", "Marketing", 70000.0, "2020-12-01", "India", 33, "Other"),
    (10, "Ivy Taylor", "Finance", 95000.0, "2017-06-30", "India", 50, "Female"),
    (11, "Jack Anderson", "Finance", 89000.0, "2021-04-18", "Australia", 38, "Male"),
    (12, "Karen Thomas", "Engineering", 81000.0, "2023-03-07", "Australia", 25, "Female"),
    (13, "Leo Moore", "Sales", 62000.0, "2019-10-25", "USA", 37, None),
    (14, "Mia Martin", "Engineering", 90000.0, "2022-05-11", "USA", 27, "Female"),
    (15, "Nick Lee", "HR", 60000.0, "2018-02-09", "UK", 42, "Male"),
    (16, "Olivia Harris", "Marketing", 74000.0, "2020-09-03", "Canada", 31, "Female"),
    (17, "Paul Clark", "Finance", 87000.0, "2023-07-19", "Germany", 24, "Male"),
    (18, "Quinn Lewis", "Engineering", 83000.0, "2021-12-12", "India", 34, "Other"),
    (19, "Rachel Walker", "Sales", 66000.0, "2019-08-08", "Australia", 36, "Female"),
    (20, "Sam Hall", "Marketing", 71000.0, "2022-11-27", "USA", 23, None),
]

## Splitting the data into two datasets of 10 rows each
emp_data_1 = emp_data[:10]
emp_data_2 = emp_data[10:]

## Creating the dataFrames
emp1 = spark.createDataFrame(
            data = emp_data_1,
            schema = emp_schema
)

emp2 = spark.createDataFrame(
            data = emp_data_2,
            schema = emp_schema
)

emp1.show()
emp2.show()

In [0]:
# UNION and UNIONALL
# combine two datasets
emp = emp1.union(emp2)
emp.show()

In [0]:
## salary in descending order
from pyspark.sql.functions import col

emp_sal = emp.orderBy(col("salary").desc())
emp_sal.show()

In [0]:
## Aggregation
# count the number of employees in each department
'''
In PySpark, data aggregation is performed by grouping identical data into subsets using the groupBy() method and then applying functions like sum, average, count, minimum, or maximum.
'''
from pyspark.sql.functions import count

emp_count = emp_sal.groupBy("department").\
                        agg(count('id').\
                            alias("total_dept_count")).\
                                orderBy(col('total_dept_count').asc())

display(emp_count)

In [0]:
# calculate total department salary
from pyspark.sql.functions import sum

emp_dept_salary = emp_sal.groupBy('department').\
                            agg(sum('salary').\
                                alias('toal_dept_sal')).\
                                    orderBy('toal_dept_sal')
                            
emp_dept_salary.show()

In [0]:
# groupBy with HAVING clause
# calculate total department avg(salary > 70k)
from pyspark.sql.functions import avg

emp_dept_salary_250k = emp_sal.groupBy('department').\
                            agg(avg('salary').\
                                alias('avg_dept_sal')).\
                                    where('avg_dept_sal > 70000').\
                                        orderBy('avg_dept_sal')
                            
emp_dept_salary_250k.show()

In [0]:
## unionByName demo
# unionByName matches columns by name (not by position), which is useful
# when the two DataFrames have columns in a different order.

schema_a = "id INT, name STRING, salary DOUBLE"
schema_b = "name STRING, id INT, salary DOUBLE"

data_a = [
    (1, "John Smith", 85000.0),
    (2, "Alice Johnson", 92000.0),
    (3, "Bob Williams", 65000.0),
]

data_b = [
    ("Carol Brown", 4, 68000.0),
    ("David Jones", 5, 55000.0),
    ("Eva Garcia", 6, 58000.0),
]

df_a = spark.createDataFrame(data=data_a, schema=schema_a)
df_b = spark.createDataFrame(data=data_b, schema=schema_b)

df_a.show()
df_b.show()

## unionByName aligns columns by name regardless of order
df_union_byname = df_a.unionByName(df_b)
df_union_byname.show()

##### For comparison, plain union() matches by position and would misalign the data, unionByName will do the better job here.

In [0]:
emp.count()

In [0]:
emp.select('department').distinct().count()